# LAB 8

## Task 1: Teoría

### 1. Investigar el algoritmo AC-3 y su relación con el algoritmo de backtracking search

El algoritmo **AC-3 (Arc Consistency Algorithm #3)** es un método utilizado en problemas de satisfacción de restricciones (CSP) para reducir el espacio de búsqueda eliminando valores inconsistentes en las variables antes o durante la búsqueda. Su objetivo principal es lograr consistencia de arco, asegurando que para cada par de variables conectadas por una restricción, los valores asignados a una variable tengan al menos un valor compatible en la otra.

La relación con el algoritmo **Backtracking Search** radica en que AC-3 generalmente se utiliza como un paso previo o complemento a la búsqueda por backtracking. Aplicar AC-3 antes de iniciar la búsqueda ayuda a disminuir considerablemente la cantidad de decisiones incorrectas que el algoritmo de backtracking podría tomar, reduciendo el tamaño del árbol de búsqueda y mejorando significativamente la eficiencia del proceso.

### 2. Defina en sus propias palabras el término “Arc Consistency”

El término **Arc Consistency (Consistencia de Arco)** en un problema CSP se refiere al estado en el cual todas las variables conectadas mediante restricciones han eliminado los valores incompatibles. En otras palabras, una variable es arc-consistente respecto a otra cuando cada uno de sus posibles valores tiene al menos una opción compatible en la variable vecina. Lograr consistencia de arco implica garantizar que ninguna variable tenga valores imposibles o incompatibles, facilitando así encontrar soluciones válidas para el problema.


In [1]:
# %% [code]
import time
import random

# Lista de exámenes y dominio de días
exams = ["Exam1", "Exam2", "Exam3", "Exam4", "Exam5", "Exam6", "Exam7"]
days = ["Lunes", "Martes", "Miércoles"]

# Inscripción de los estudiantes en los exámenes
students = {
    "A": ["Exam1", "Exam2", "Exam3"],
    "B": ["Exam2", "Exam4", "Exam5"],
    "C": ["Exam3", "Exam5", "Exam6", "Exam7"],
    "D": ["Exam1", "Exam4", "Exam6"]
}

# Construir el grafo de conflictos: si dos exámenes son tomados por el mismo estudiante, no pueden ser el mismo día.
conflicts = {exam: set() for exam in exams}
for student, course_list in students.items():
    for i in range(len(course_list)):
        for j in range(i + 1, len(course_list)):
            exam_i = course_list[i]
            exam_j = course_list[j]
            conflicts[exam_i].add(exam_j)
            conflicts[exam_j].add(exam_i)

# Función para verificar consistencia de una asignación parcial.
def is_consistent(assignment, var, value):
    # Para cada examen que entra en conflicto con 'var', si ya está asignado el mismo día, se viola la restricción.
    for neighbor in conflicts[var]:
        if neighbor in assignment and assignment[neighbor] == value:
            return False
    return True

# Función auxiliar para mostrar la información de conflictos (opcional)
def print_conflicts():
    for exam, conflict_set in conflicts.items():
        print(f"{exam} entra en conflicto con: {conflict_set}")

#print_conflicts()

# %% [code]
def backtracking(assignment):
    # Si se han asignado todos los exámenes, se ha encontrado una solución.
    if len(assignment) == len(exams):
        return assignment
    # Seleccionamos la siguiente variable (examen) sin asignar.
    unassigned = [e for e in exams if e not in assignment]
    var = unassigned[0]
    for value in days:
        if is_consistent(assignment, var, value):
            assignment[var] = value
            result = backtracking(assignment)
            if result is not None:
                return result
            del assignment[var]  # Retroceso
    return None

# Medición del tiempo para Backtracking
start_bt = time.time()
bt_solution = backtracking({})
bt_time = time.time() - start_bt

print("Backtracking solution:", bt_solution)
print("Tiempo (segundos):", bt_time)

# %% [code]
def beam_search(beam_width):
    # Cada estado es una asignación parcial (diccionario).
    initial_state = {}
    beam = [initial_state]
    while beam:
        new_beam = []
        for state in beam:
            # Si la asignación está completa, se retorna la solución.
            if len(state) == len(exams):
                return state
            # Elegimos la siguiente variable sin asignar (orden simple)
            unassigned = [e for e in exams if e not in state]
            var = unassigned[0]
            for value in days:
                if is_consistent(state, var, value):
                    new_state = state.copy()
                    new_state[var] = value
                    new_beam.append(new_state)
        # Si no se pueden generar más estados, se falla.
        if not new_beam:
            return None
        # Se seleccionan los mejores estados: en este caso, los que tienen mayor cantidad de asignaciones.
        new_beam.sort(key=lambda s: len(s), reverse=True)
        beam = new_beam[:beam_width]
    return None

# Medición del tiempo para Beam Search con un beam_width de 3
start_beam = time.time()
beam_solution = beam_search(beam_width=3)
beam_time = time.time() - start_beam

print("Beam Search solution:", beam_solution)
print("Tiempo (segundos):", beam_time)

# %% [code]
def count_conflicts(assignment, var, value):
    # Cuenta el número de conflictos que se producirían al asignar 'value' a 'var'.
    count = 0
    for neighbor in conflicts[var]:
        if neighbor in assignment and assignment[neighbor] == value:
            count += 1
    return count

def min_conflicts(max_iter=1000):
    # Se crea una asignación completa aleatoria.
    assignment = {exam: random.choice(days) for exam in exams}
    for i in range(max_iter):
        # Se identifican los exámenes que están en conflicto.
        conflicted = []
        for exam in exams:
            if any(assignment[exam] == assignment[neighbor] for neighbor in conflicts[exam] if neighbor in assignment):
                conflicted.append(exam)
        # Si no hay conflictos, se ha encontrado la solución.
        if not conflicted:
            return assignment
        # Se selecciona aleatoriamente un examen en conflicto.
        var = random.choice(conflicted)
        # Se selecciona el valor del dominio que minimice los conflictos.
        best_value = None
        best_conflict = float('inf')
        for value in days:
            c = count_conflicts(assignment, var, value)
            if c < best_conflict:
                best_conflict = c
                best_value = value
        assignment[var] = best_value
    return None

# Medición del tiempo para Local Search
start_local = time.time()
local_solution = min_conflicts(max_iter=1000)
local_time = time.time() - start_local

print("Local Search solution:", local_solution)
print("Tiempo (segundos):", local_time)


Backtracking solution: None
Tiempo (segundos): 0.00016427040100097656
Beam Search solution: None
Tiempo (segundos): 8.320808410644531e-05
Local Search solution: None
Tiempo (segundos): 0.01167440414428711
